# 00 · Data preparation

Runs on CPU. Reads the ASVspoof 2019 LA dataset, verifies it, and writes everything the training and evaluation notebooks read:

| Output | Contents |
|---|---|
| `manifest.csv` | One row per protocol utterance: split, speaker, attack, label, length, edge silence, cache offset |
| `cache/{train,dev}_int16.bin` | Decoded int16 audio for train and dev, concatenated; located via `cache_offset` and `n_samples` |
| `feature_stats.json` | Per-channel mel / Δ / ΔΔ normalisation statistics from the full train split |
| `provenance.json` | Hashes of protocols and caches, integrity results, library versions |

Every FLAC is checked against the MD5 of its decoded audio stored in its own header, so corruption anywhere between the original download and Kaggle is detected without an external source.

In [1]:
import hashlib
import json
import os
import platform
import sys
import time
from multiprocessing import Pool

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torchaudio

INPUT_ROOT = "/kaggle/input"
WORK_DIR = "/kaggle/working"
CACHE_SPLITS = ("train", "dev")
N_WORKERS = os.cpu_count()
STATS_BATCH = 64
SEED = 0

In [2]:
def walk_without_audio(root):
    for dirpath, dirnames, filenames in os.walk(root, followlinks=True):
        dirnames[:] = sorted(d for d in dirnames if d != "flac")
        yield dirpath, dirnames, filenames


try:
    import audio_utils as U
except ImportError:
    found = [os.path.join(d, "audio_utils.py") for d, _, f in walk_without_audio(INPUT_ROOT) if "audio_utils.py" in f]
    if len(found) != 1:
        raise FileNotFoundError(f"expected one audio_utils.py under {INPUT_ROOT}, found {found}")
    sys.path.insert(0, os.path.dirname(found[0]))
    import audio_utils as U

U._self_test()

roots = [d for d, sub, _ in walk_without_audio(INPUT_ROOT) if "ASVspoof2019_LA_cm_protocols" in sub]
if len(roots) != 1:
    raise FileNotFoundError(f"expected one LA root under {INPUT_ROOT}, found {roots}")
LA_ROOT = roots[0]
LA_ROOT

audio_utils 1.0.1 self-test passed (sha 658e72bce982)


'/kaggle/input/datasets/rudranshiverma/asv-spoof-la-track/data'

## Protocols

Parsing fails unless each split matches the official composition (bona fide / spoof counts and attack IDs) and no utterance or speaker appears in two splits.

In [3]:
proto_dir = os.path.join(LA_ROOT, "ASVspoof2019_LA_cm_protocols")
frames = {s: U.parse_cm_protocol(os.path.join(proto_dir, U.PROTOCOL_FILES[s]), s) for s in U.SPLITS}
U.assert_disjoint_splits(frames)

manifest = pd.concat(frames.values(), ignore_index=True)
manifest["rel_path"] = [f"{U.AUDIO_DIRS[s]}/flac/{u}.flac" for s, u in zip(manifest["split"], manifest["utt_id"])]
protocol_sha256 = {s: U.file_sha256(os.path.join(proto_dir, f)) for s, f in U.PROTOCOL_FILES.items()}

pd.crosstab(manifest["attack"], manifest["split"]).loc[:, list(U.SPLITS)]

split,train,dev,eval
attack,,,
A01,3800,3716,0
A02,3800,3716,0
A03,3800,3716,0
A04,3800,3716,0
A05,3800,3716,0
A06,3800,3716,0
A07,0,0,4914
A08,0,0,4914
A09,0,0,4914


## Audio integrity and cache

Each file must decode as 16 kHz mono 16-bit PCM and its decoded samples must match the MD5 in the FLAC header. Train and dev audio is appended to the cache in manifest order.

In [4]:
def flac_header_md5(path):
    with open(path, "rb") as fh:
        head = fh.read(42)
    if head[:4] != b"fLaC" or head[4] & 0x7F != 0:
        raise ValueError("missing FLAC STREAMINFO block")
    return head[26:42]


def inspect(job):
    rel, keep_audio = job
    path = os.path.join(LA_ROOT, rel)
    try:
        x = U.read_audio_int16(path)
        stored = flac_header_md5(path)
        if stored == bytes(16):
            md5 = "unset"
        elif hashlib.md5(x.astype("<i2").tobytes()).digest() == stored:
            md5 = "ok"
        else:
            return rel, "decoded audio does not match header MD5", None
        lead, trail = U.edge_silence(x)
        out = {"n_samples": int(x.size), "lead_silence": lead, "trail_silence": trail, "md5": md5}
        return rel, None, (out, x if keep_audio else None)
    except Exception as e:
        return rel, f"{type(e).__name__}: {e}", None


os.makedirs(os.path.join(WORK_DIR, "cache"), exist_ok=True)
cache_paths = {s: os.path.join(WORK_DIR, "cache", f"{s}_int16.bin") for s in CACHE_SPLITS}
split_of = dict(zip(manifest["rel_path"], manifest["split"]))
jobs = [(r, split_of[r] in CACHE_SPLITS) for r in manifest["rel_path"]]

records, errors = {}, []
offsets = {s: 0 for s in CACHE_SPLITS}
handles = {s: open(p, "wb") for s, p in cache_paths.items()}
t0 = time.time()
with Pool(N_WORKERS) as pool:
    for i, (rel, err, res) in enumerate(pool.imap(inspect, jobs, chunksize=32), start=1):
        if i % (len(jobs) // 10) == 0:
            print(f"{i}/{len(jobs)} files, {time.time() - t0:.0f} s")
        if err:
            errors.append((rel, err))
            continue
        out, audio = res
        split = split_of[rel]
        out["cache_offset"] = -1
        if audio is not None:
            out["cache_offset"] = offsets[split]
            handles[split].write(audio.astype("<i2").tobytes())
            offsets[split] += audio.size
        records[rel] = out
for h in handles.values():
    h.close()
print(f"{len(jobs)} files in {time.time() - t0:.0f} s")

if errors:
    for rel, err in errors[:20]:
        print(rel, "->", err)
    raise RuntimeError(f"{len(errors)} files failed integrity checks")

manifest = manifest.join(pd.DataFrame.from_dict(records, orient="index"), on="rel_path")
if manifest[["n_samples", "cache_offset", "md5"]].isna().any().any():
    raise AssertionError("manifest has rows without audio records")
manifest["n_samples"] = manifest["n_samples"].astype(np.int64)
manifest["cache_offset"] = manifest["cache_offset"].astype(np.int64)
manifest["md5"].value_counts()

12146/121461 files, 37 s
24292/121461 files, 72 s
36438/121461 files, 117 s
48584/121461 files, 157 s
60730/121461 files, 195 s
72876/121461 files, 231 s
85022/121461 files, 265 s
97168/121461 files, 300 s
109314/121461 files, 334 s
121460/121461 files, 367 s
121461 files in 367 s


md5
ok    121461
Name: count, dtype: int64

In [5]:
manifest["seconds"] = manifest["n_samples"] / U.TARGET_SR
summary = manifest.groupby(["split", "key"]).agg(
    clips=("utt_id", "size"),
    median_s=("seconds", "median"),
    max_s=("seconds", "max"),
    over_4s=("seconds", lambda s: (s > 4).mean()),
    lead_sil_s=("lead_silence", lambda s: s.mean() / U.TARGET_SR),
    trail_sil_s=("trail_silence", lambda s: s.mean() / U.TARGET_SR),
)
summary.round(3)

clips  median_s   max_s  over_4s  lead_sil_s  trail_sil_s
split key                                                                
dev   bonafide   2548     3.387  11.389    0.301       0.904        0.404
      spoof     22296     3.266  11.594    0.299       0.500        0.329
eval  bonafide   7355     3.196  13.010    0.235       0.813        0.434
      spoof     63882     2.753  13.026    0.221       0.184        0.217
train bonafide   2580     3.240  11.129    0.235       0.728        0.327
      spoof     22800     3.197  13.188    0.280       0.503        0.323

## Normalisation statistics

Exact per-channel mean and standard deviation over every value of every train clip, after the recipe's first-4-seconds crop and zero padding, so the statistics describe exactly what the model sees.

In [6]:
torch.set_num_threads(N_WORKERS)
cache = {s: np.memmap(p, dtype="<i2", mode="r") for s, p in cache_paths.items()}
for s, mm in cache.items():
    if mm.size != manifest.loc[manifest["split"] == s, "n_samples"].sum():
        raise AssertionError(f"{s} cache length does not match manifest")


def batch_waves(rows, split):
    mm = cache[split]
    return torch.stack([
        U.fit_head(torch.from_numpy(U.int16_to_float32(np.array(mm[o:o + n]))))
        for o, n in zip(rows["cache_offset"], rows["n_samples"])
    ])


fx = U.LogMelDelta()
acc = U.StatsAccumulator()
train = manifest[manifest["split"] == "train"]
for i in range(0, len(train), STATS_BATCH):
    acc.update(fx.raw(batch_waves(train.iloc[i:i + STATS_BATCH], "train")))
stats = acc.result()
pd.DataFrame({"mean": stats.mean, "std": stats.std}, index=U.CHANNELS)

,mean,std
mel,-37.969225,38.371481
delta,-0.145588,3.897335
delta2,-0.000213,1.639660


## Checks and outputs

The cache must reproduce a random sample of files exactly, and normalised train features must have zero mean and unit variance per channel. Dev is reported, not asserted.

In [7]:
sample = manifest[manifest["split"].isin(CACHE_SPLITS)].sample(500, random_state=SEED)
for r in sample.itertuples():
    cached = np.array(cache[r.split][r.cache_offset:r.cache_offset + r.n_samples])
    if not np.array_equal(cached, U.read_audio_int16(os.path.join(LA_ROOT, r.rel_path))):
        raise AssertionError(f"cache mismatch for {r.utt_id}")

norm = U.LogMelDelta(stats)
U.check_feature_batch(norm(batch_waves(train.sample(256, random_state=SEED), "train")))
dev_feats = norm(batch_waves(manifest[manifest["split"] == "dev"].sample(256, random_state=SEED), "dev"))
pd.DataFrame({
    "dev_mean": dev_feats.mean(dim=(0, 2, 3)).numpy(),
    "dev_std": dev_feats.std(dim=(0, 2, 3)).numpy(),
}, index=U.CHANNELS).round(3)

,dev_mean,dev_std
mel,-0.032,1.012
delta,-0.004,0.979
delta2,-0.000,0.980


In [8]:
cols = ["utt_id", "split", "speaker", "attack", "key", "label", "rel_path",
        "n_samples", "lead_silence", "trail_silence", "cache_offset", "md5"]
manifest[cols].to_csv(os.path.join(WORK_DIR, "manifest.csv"), index=False)

with open(os.path.join(WORK_DIR, "feature_stats.json"), "w") as fh:
    json.dump(stats.to_dict(), fh, indent=2)

provenance = {
    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "la_root": LA_ROOT,
    "utils_version": U.UTILS_VERSION,
    "utils_sha256": U.utils_sha256(),
    "protocol_sha256": protocol_sha256,
    "manifest_sha256": U.file_sha256(os.path.join(WORK_DIR, "manifest.csv")),
    "feature_stats_sha256": U.file_sha256(os.path.join(WORK_DIR, "feature_stats.json")),
    "cache": {s: {"file": os.path.relpath(p, WORK_DIR), "samples": int(cache[s].size), "sha256": U.file_sha256(p)}
              for s, p in cache_paths.items()},
    "flac_md5": manifest["md5"].value_counts().to_dict(),
    "counts": {s: frames[s]["key"].value_counts().to_dict() for s in U.SPLITS},
    "versions": {
        "python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
        "torch": torch.__version__, "torchaudio": torchaudio.__version__, "soundfile": sf.__version__,
    },
}
with open(os.path.join(WORK_DIR, "provenance.json"), "w") as fh:
    json.dump(provenance, fh, indent=2)

print(json.dumps({k: provenance[k] for k in ("flac_md5", "counts", "cache")}, indent=2))

{
  "flac_md5": {
    "ok": 121461
  },
  "counts": {
    "train": {
      "spoof": 22800,
      "bonafide": 2580
    },
    "dev": {
      "spoof": 22296,
      "bonafide": 2548
    },
    "eval": {
      "spoof": 63882,
      "bonafide": 7355
    }
  },
  "cache": {
    "train": {
      "file": "cache/train_int16.bin",
      "samples": 1391158676,
      "sha256": "d2757bbdc975807605fd2124b9a6d0daebd4f0a1f911f165d32a19c0f4fdb433"
    },
    "dev": {
      "file": "cache/dev_int16.bin",
      "samples": 1382564017,
      "sha256": "638063326750c2e6ee3eaf765795a36221f305e25e0de784d26341c753460603"
    }
  }
}
